# Lenormand B4-Q38F — Full64 三折 OOF 完成版

Fold 0 已经证明完整 Qwen3.8-27B 有真实潜力：Macro-AP `0.7668`，明显超过同折 Qwen3-14B 的 `0.7284`。本 notebook 不再做微型筛选，而是完成真正的模型决策：

1. **只续跑 Fold 1 / Fold 2**，已完成的 Fold 0 不会重训；
2. 每折训练、评分均可从 Google Drive checkpoint / chunk 自动恢复；
3. 将三折 held-out logits 拼成完整 OOF，严格检查 row、fold、target 与缺失值；
4. 在完全相同的 grouped folds 上比较 14B、27B 与非负逐标签融合；
5. 单独报告 Fold 1/2 的 confirmation 结果，避免把用于选择方案的 Fold 0 假装成独立确认。

在 A100 80GB + FlashAttention/FLA 已生效的情况下，**剩余两折通常约 6–8 小时**。中途断线后不要删除结果目录，重跑同一格即可续上。


In [ ]:
#@title 0A. 安装基础依赖
%%capture
!pip install -q -U   "transformers>=5.8.0"   "accelerate>=1.6.0"   "peft>=0.17.0"   "bitsandbytes>=0.46.0"   "sentence-transformers>=3.4.0"   "sentencepiece>=0.2.0"   "openpyxl>=3.1.0"   "scikit-learn>=1.5.0"   "scipy>=1.13.0"   "kernels"


In [ ]:
#@title 0B. 安装 Qwen3.8 混合架构 kernels
!pip install -U "flash-linear-attention[cuda]"
!pip install -U causal-conv1d --no-build-isolation

print('安装完成。若这是新 runtime：Runtime → Restart session，然后从第 1 格继续。')


> 新 runtime 必须重启一次。否则 Qwen3.8 可能缓存“kernel 不可用”的旧状态。重启后从第 1 格开始，不要重跑 0A/0B。


In [ ]:
#@title 1. Drive、路径与续跑开关
from google.colab import drive, files
drive.mount('/content/drive')

from pathlib import Path
import dataclasses, gc, importlib, json, math, shutil, subprocess, sys, time

ROOT = Path('/content/drive/MyDrive/IEEE_BigData2026')
TRAIN_PATH = ROOT / 'train.xlsx'

# 与已经完成的 Fold 0 使用同一根目录，避免复制 56GB 模型或丢失缓存。
Q38_RUN_ROOT = ROOT / 'results' / 'B4_Q38F_FULL64_KERNEL_FOLD0'
FULL64_FOLD_ROOT = Q38_RUN_ROOT / 'FULL64_FACTOR_FOLD0'
OOF_REPORT_ROOT = ROOT / 'results' / 'B4_Q38F_FULL64_THREE_FOLD_OOF'
OLD_FAST_ROOT = ROOT / 'results' / 'QWEN38_DUAL_FOLD0_FAST_LAST16'

B1_PATH = ROOT / 'b1_experiments.py'
B4_PATH = ROOT / 'b4p_anchor_verifier.py'
Q38_PATH = ROOT / 'qwen38_dual_task_experiments.py'
OOF_HELPER_PATH = ROOT / 'b4_q38_full_oof.py'
EXISTING_Q14_OOF = ROOT / 'results' / 'B4P_AVC_FAST3' / 'B4P_CORE_OOF.npz'

# 正常续跑就是 (1, 2)。某折完成后再次运行会直接复用，不会重训。
RUN_FOLDS = (1, 2)
RUN_COMPLETE_OOF = True
OVERWRITE_ADAPTERS = False       # 永远不要在断线恢复时改成 True

OOF_REPORT_ROOT.mkdir(parents=True, exist_ok=True)
required = {
    B1_PATH: None,
    B4_PATH: 'B4P_RUNTIME_REVISION = "2026-08-21.qwen38-full64-kernels-v4"',
    Q38_PATH: None,
    OOF_HELPER_PATH: 'Q38_OOF_REVISION = "2026-08-23.full64-three-fold-oof-v2"',
}
stale = [path for path, marker in required.items()
         if (not path.exists()) or (marker is not None and marker not in path.read_text(encoding='utf-8'))]
if stale:
    print('请上传本次模块并覆盖 Drive：', [path.name for path in stale])
    uploaded = files.upload()
    for path in stale:
        if path.name not in uploaded:
            raise FileNotFoundError(path)
        shutil.copy2('/content/' + path.name, path)

assert TRAIN_PATH.exists(), TRAIN_PATH
assert EXISTING_Q14_OOF.exists(), EXISTING_Q14_OOF
print(subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True,
).stdout)
drive_free_gb = shutil.disk_usage(ROOT).free / 2**30
print(f'Drive free: {drive_free_gb:.1f} GB')
if drive_free_gb < 12:
    print('WARNING: 两折 checkpoint 可能用满 Drive；建议至少保留 12 GB。')
print('Fold artifacts:', FULL64_FOLD_ROOT)
print('OOF report:', OOF_REPORT_ROOT)


In [ ]:
#@title 2. Kernel 硬检查与完全复刻 Fold 0 的 Full64 配置
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import torch
import transformers
import b1_experiments as b1
import b4p_anchor_verifier as b4
import qwen38_dual_task_experiments as q38
import b4_q38_full_oof as q38oof
importlib.reload(b1); importlib.reload(b4); importlib.reload(q38); importlib.reload(q38oof)

assert b4.B4P_RUNTIME_REVISION == '2026-08-21.qwen38-full64-kernels-v4'
assert q38oof.Q38_OOF_REVISION == '2026-08-23.full64-three-fold-oof-v2'
kernel_status = b4.qwen35_kernel_status()
print('transformers:', transformers.__version__)
print('torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('kernel status:', kernel_status)
assert kernel_status['causal_conv1d'], 'causal_conv1d 未生效：确认安装后重启过 runtime'
assert kernel_status['flash_linear_attention'], 'FLA 未生效：确认安装后重启过 runtime'
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
assert gpu_memory_gb >= 70, f'Full64 配置要求 A100 80GB；当前只有 {gpu_memory_gb:.1f} GB'

MODEL = 'Qwen/Qwen3.8-27B'
CFG = b4.B4PConfig(
    seed=42,
    n_splits=3,
    verifier_model=MODEL,
    max_length=1536,
    context_char_budget=5000,
    full_post_char_limit=4300,
    context_top_clauses=8,
    retrieval_example_chars=600,
    include_retrieval=True,
    verifier_use_chat_template=True,
    prompt_truncation_side='left',
    attention_implementation='flash_attention_2',
    qwen35_fa2_position_guard=True,
    require_qwen35_fast_kernels=True,
    positives_floor=64,
    examples_cap_per_class=512,
    negative_ratio=1.0,
    hard_negative_fraction=0.7,
    sft_epochs=1.0,
    sft_learning_rate=1.0e-4,
    sft_batch_size=1,
    sft_gradient_accumulation=32,
    lora_r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    lora_last_n_layers=None,
    lora_target_leaves=(
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
        'in_proj_qkv', 'in_proj_z', 'in_proj_a', 'in_proj_b', 'out_proj',
    ),
    gradient_checkpointing=True,
    verifier_score_batch_size=2,
    score_chunk_size=256,
    threshold_kappa_tail=0.0,
    threshold_kappa_mid=2.0,
    threshold_kappa_head=2.0,
    stack_l2=0.02,
)

b4.seed_everything(CFG.seed)
torch.set_float32_matmul_precision('high')
(OOF_REPORT_ROOT / 'FULL64_CONFIG.json').write_text(
    json.dumps(dataclasses.asdict(CFG), indent=2), encoding='utf-8'
)
print('Full64 config locked. attention=', CFG.attention_implementation,
      'last_n_layers=', CFG.lora_last_n_layers)


In [ ]:
#@title 3. 数据、相同 folds、语义缓存与 Fold 0 完整性
bundle = b1.load_training_data(ROOT, TRAIN_PATH)
q14_logits, folds, targets = q38oof.load_q14_oof(EXISTING_Q14_OOF, bundle)
print('fold sizes:', np.bincount(folds).tolist())
print(b4.validate_folds(bundle, folds))

fold0_path = q38oof.q38_fold_path(FULL64_FOLD_ROOT, 0)
assert fold0_path.exists(), f'找不到已完成 Fold 0：{fold0_path}'
fold0_saved = np.load(fold0_path, allow_pickle=True)
assert np.isfinite(fold0_saved['logits'][folds == 0]).all()
print('Fold 0 ready:', fold0_path)

corpus = b4.training_corpus(bundle)
cache_candidates = [
    OLD_FAST_ROOT / 'semantic_cache' / 'train',
    Q38_RUN_ROOT / 'semantic_cache' / 'train',
]
existing_cache = next((path for path in cache_candidates if path.exists()), cache_candidates[-1])
semantic_cache = b4.prepare_semantic_cache(corpus, CFG, existing_cache)
print('Semantic cache:', existing_cache)


In [ ]:
#@title 4. 续跑前状态审计（不会加载 27B）
status_rows = []
for fold in range(CFG.n_splits):
    fold_dir = FULL64_FOLD_ROOT / f'fold_{fold}'
    logits_path = q38oof.q38_fold_path(FULL64_FOLD_ROOT, fold)
    adapter_path = fold_dir / 'verifier' / 'adapter_final' / 'adapter_config.json'
    checkpoints = sorted((fold_dir / 'verifier' / 'checkpoints').glob('checkpoint-*'))
    complete_logits = False
    if logits_path.exists():
        saved = np.load(logits_path, allow_pickle=True)
        complete_logits = bool(np.isfinite(saved['logits'][folds == fold]).all())
    train_rows = np.flatnonzero(folds != fold)
    manifest_path = fold_dir / 'sft_pair_manifest.csv'
    if manifest_path.exists():
        n_pairs = len(pd.read_csv(manifest_path))
    else:
        n_pairs = len(b4.build_pair_manifest(bundle, semantic_cache, train_rows, CFG, fold))
    status_rows.append({
        'fold': fold,
        'heldout_rows': int((folds == fold).sum()),
        'training_pairs': n_pairs,
        'optimizer_updates_est': math.ceil(n_pairs / CFG.sft_gradient_accumulation),
        'adapter_complete': adapter_path.exists(),
        'logits_complete': complete_logits,
        'latest_checkpoint': checkpoints[-1].name if checkpoints else None,
        'scheduled_now': fold in RUN_FOLDS,
    })
status = pd.DataFrame(status_rows)
display(status)
print('预计剩余：每个未完成 fold 约 3–4 小时；两个 fold 约 6–8 小时。')


In [ ]:
#@title 5. 依次完成 Fold 1 / Fold 2（断线后重跑本格）
fold_run_records = []
for fold in RUN_FOLDS:
    logits_path = q38oof.q38_fold_path(FULL64_FOLD_ROOT, fold)
    if logits_path.exists():
        saved = np.load(logits_path, allow_pickle=True)
        if np.isfinite(saved['logits'][folds == fold]).all():
            metrics_path = logits_path.with_name('qwen38_factor_fold_metrics.json')
            metrics = json.loads(metrics_path.read_text()) if metrics_path.exists() else {'fold': fold}
            print(f'[Fold {fold}] complete，直接复用 {logits_path}')
            fold_run_records.append({'fold': fold, 'resumed_complete': True, **metrics})
            continue

    print(f'\n========== START / RESUME FULL64 FOLD {fold} ==========')
    started = time.perf_counter()
    metrics, adapter = q38.run_qwen38_factor_fold(
        bundle, folds, semantic_cache, CFG, fold,
        FULL64_FOLD_ROOT, overwrite=OVERWRITE_ADAPTERS,
    )
    elapsed_hours = (time.perf_counter() - started) / 3600
    fold_run_records.append({
        'fold': fold, 'resumed_complete': False,
        'elapsed_hours_this_session': elapsed_hours, **metrics,
    })
    print(f'[Fold {fold}] done in this session: {elapsed_hours:.2f} h')
    gc.collect(); torch.cuda.empty_cache()

display(pd.DataFrame(fold_run_records))
print('Fold 1/2 若都显示 complete，即可运行第 6 格。')


In [ ]:
#@title 6. 拼接完整三折 OOF + 14B/27B/融合正式决策
decision = None
if RUN_COMPLETE_OOF:
    q38_logits, artifact_manifest = q38oof.load_q38_full_oof(
        FULL64_FOLD_ROOT, bundle, folds, n_splits=CFG.n_splits,
    )
    artifact_manifest.to_csv(OOF_REPORT_ROOT / 'Q38_FOLD_ARTIFACT_MANIFEST.csv', index=False)
    np.savez_compressed(
        OOF_REPORT_ROOT / 'Q38_FULL64_OOF.npz',
        row_ids=bundle.row_ids,
        folds=folds,
        targets=targets,
        logits=q38_logits,
    )
    decision = q38oof.run_full_oof_decision(
        bundle, folds, q14_logits, q38_logits, CFG,
        OOF_REPORT_ROOT / 'OOF_EVALUATION',
    )

    print('\n=== PRIMARY OOF SUMMARY ===')
    display(pd.read_csv(
        OOF_REPORT_ROOT / 'OOF_EVALUATION' / 'Q38_FULL_OOF_SUMMARY.csv'
    ))
    print('\n=== FOLD 1/2 CONFIRMATION（Fold 0 只是 screening）===')
    display(pd.read_csv(
        OOF_REPORT_ROOT / 'OOF_EVALUATION' / 'Q38_FOLD_CONFIRMATION.csv'
    ).sort_values(['fold', 'model']))
    print('\n=== PER-LABEL GAINS ===')
    per_label = pd.read_csv(
        OOF_REPORT_ROOT / 'OOF_EVALUATION' / 'Q38_PER_LABEL_OOF_COMPARISON.csv'
    )
    display(per_label.head(12))
    print('\n=== PER-LABEL REGRESSIONS ===')
    display(per_label.tail(12).sort_values('delta_ap'))
    print('\n=== DECISION ===')
    print(json.dumps(decision, indent=2, default=str))
else:
    print('RUN_COMPLETE_OOF=False')


In [ ]:
#@title 7. 打包轻量报告下载（不包含巨大 adapter/checkpoints）
if decision is None:
    raise RuntimeError('请先完成第 6 格')

archive_base = Path('/content/B4_Q38F_FULL64_THREE_FOLD_OOF_REPORT')
archive_path = shutil.make_archive(
    str(archive_base), 'zip', root_dir=OOF_REPORT_ROOT,
)
print('Report ZIP:', archive_path)
print('推荐系统:', decision['recommended_factor_system'])
print('27B accepted:', decision['q38_standalone_accepted'])
print('Stack accepted:', decision['stack_accepted'])

# 需要下载时取消下一行注释；adapter/checkpoint 已安全留在 Drive。
# files.download(archive_path)


## 如何读最终结果

- `QWEN38_27B_FULL64`：27B 单模，是否真的打赢完整 14B OOF；这是最可信的升级判断。
- `B4P_NONNEGATIVE_STACK`：逐标签、非负、cross-fitted 融合。如果同时超过最佳单模，才进入最终候选。
- 三个 `EXPLORATORY_PROB_BLEND_*`：用于观察互补性；因为 0.75 权重受到 Fold 0 结果启发，不允许它单独推翻预先写入代码的 acceptance gate。
- `Q38_FOLD_CONFIRMATION.csv`：重点看 Fold 1 和 Fold 2。至少一个 confirmation fold 的 AP 必须赢，且完整 OOF 的 Macro-F1 / AP / tail 安全条件同时通过，27B 才正式晋级。

这一本先决定 Factor 主干。确认胜者后再生成 final-train + test submission notebook，避免在尚未确认的 27B 上继续烧三套测试推理。
